# 02 — Différencier les FRAPPES : bass / tone / slap

Les 3 frappes de base du jembé : **basse** (grave, paume au centre), **tonique** (médium résonant, doigts au bord), **claqué/slap** (aigu, sec, bruité).

On travaille sur les **jembés isolés** du corpus (J1 soliste + J2 accompagnement), donc du son **propre** et calé sur les **onsets de référence**. Pas d'étiquette de frappe dans le corpus → **clustering non-supervisé**, validé **à l'oreille**.

## 0. Imports & jeu de frappes (avec cache)

In [ ]:
import os, warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import umap
import djembe_utils as dj

CACHE = 'output/jembe_strokes.csv'
if os.path.exists(CACHE):
    df = pd.read_csv(CACHE)
    print('Cache chargé :', len(df), 'frappes')
else:
    c = dj.list_corpus(); jem = c[c['role'].str.startswith('jembe')]
    df = pd.concat([dj.strokes_dataframe(r['piece'], r['take'], r['code'], max_per_track=150)
                    for _, r in jem.iterrows()], ignore_index=True)
    os.makedirs('output', exist_ok=True); df.to_csv(CACHE, index=False)
    print('Construit + mis en cache :', len(df), 'frappes')

## 1. Clustering en 3 types

KMeans à k=3. **Étiquetage physique** (le centroïde seul est trompeur) : la **basse** = max d'énergie grave (`low_ratio`), le **slap** = max de bruit (`flatness`), la **tonique** = le reste.

In [ ]:
X = StandardScaler().fit_transform(df[dj.FEAT_COLS])
for k in range(2, 6):
    lab = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    print(f'  k={k}: silhouette={silhouette_score(X, lab, sample_size=4000, random_state=0):.3f}')

df['cl'] = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X)
g = df.groupby('cl').agg(low_ratio=('low_ratio','mean'), flatness=('flatness','mean'))
bass = g['low_ratio'].idxmax()
slap = g.drop(bass)['flatness'].idxmax()
tone = [i for i in g.index if i not in (bass, slap)][0]
df['stroke'] = df['cl'].map({bass:'bass', tone:'tone', slap:'slap'})
print()
print(df.groupby('stroke').agg(low_ratio=('low_ratio','mean'), flatness=('flatness','mean'),
      zcr=('zcr','mean'), centroid=('centroid','mean'), n=('stroke','size')).round(3))

## 2. Écouter les 3 types (validation à l'oreille)
On enchaîne des frappes de chaque type, prises directement sur les pistes isolées. La basse doit sonner grave, la tonique claire, le slap sec/claquant.

In [ ]:
_tracks = {}
def get_track(piece, take, code):
    key = (piece, int(take), code)
    if key not in _tracks:
        _tracks[key] = dj.load_track(piece, int(take), code)
    return _tracks[key]

def ecouter(stroke, n=14, dur=0.25):
    sub = df[df['stroke']==stroke].sample(min(n, (df['stroke']==stroke).sum()), random_state=1)
    clips = []
    for _, r in sub.iterrows():
        y = get_track(r['piece'], r['take'], r['code']); s = int(r['time']*dj.SR)
        clips.append(y[s:s+int(dur*dj.SR)])
    print(f'>>> {stroke}'); display(Audio(np.concatenate(clips), rate=dj.SR))

for s in ['bass', 'tone', 'slap']:
    ecouter(s)

## 3. Carte des frappes (UMAP, façon « oiseaux »)

In [ ]:
emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X)
df['ux'], df['uy'] = emb[:,0], emb[:,1]
COL = {'bass':'#2b6cff', 'tone':'#d63bd6', 'slap':'#2ec45f'}

plt.figure(figsize=(10,8), facecolor='#0d0d0d'); ax=plt.gca(); ax.set_facecolor('#0d0d0d')
for s,c in COL.items():
    m = df['stroke']==s
    plt.scatter(df.loc[m,'ux'], df.loc[m,'uy'], s=7, c=c, alpha=.6, label=s)
plt.legend(markerscale=3, labelcolor='white', facecolor='#1a1a1a'); ax.tick_params(colors='#555')
plt.title('Frappes de jembé : bass / tone / slap (UMAP)', color='white')
plt.savefig('output/02_frappes_umap.png', dpi=140, facecolor='#0d0d0d', bbox_inches='tight'); plt.show()

## 4. Bonus — les frappes selon le rôle
Le soliste (J1) et l'accompagnateur (J2) n'utilisent pas les mêmes frappes. Indice direct pour l'étape **rôles** (`03_roles`).

In [ ]:
ct = pd.crosstab(df['role'], df['stroke'], normalize='index')[['bass','tone','slap']]
print((ct*100).round(0).astype(int))
ct.plot.bar(stacked=True, color=[COL[s] for s in ['bass','tone','slap']], figsize=(7,4))
plt.ylabel('proportion'); plt.title('Répartition des frappes par rôle'); plt.xticks(rotation=0)
plt.legend(title='frappe'); plt.tight_layout(); plt.show()

## Bilan & suite

- 3 types de frappes obtenus par clustering, étiquetés par leurs caractéristiques physiques, **vérifiables à l'oreille** (section 2).
- Le silhouette est modeste (~0.2) : les frappes forment un **continuum** plus que 3 paquets nets — normal pour du jeu réel.
- Découverte : soliste ↔ beaucoup de **slaps**, accompagnement ↔ beaucoup de **tones**. → utile pour `03_roles`.
- On sauvegarde les frappes étiquetées.

In [ ]:
df.to_csv('output/jembe_strokes_labelled.csv', index=False)
print('Sauvegardé -> output/jembe_strokes_labelled.csv |', len(df), 'frappes')